# vehicle1: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('31_vehicle1_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,55.440,43.888,65.525,33.129,94.119,63.624,65.028,85.311,49.664
smote,81.485,67.940,55.332,88.245,75.360,81.803,68.137,88.596,74.712
bsmote,81.557,67.964,55.083,88.953,74.882,81.918,66.879,88.311,74.761
ros,79.338,65.822,55.163,82.653,76.949,79.801,67.377,88.270,72.580
adasyn,81.979,68.287,54.807,90.766,74.164,82.465,68.120,89.028,75.133
casb,80.932,67.212,54.566,87.542,74.881,81.211,68.701,88.475,74.072
casb_relaxed,81.448,67.913,55.400,87.775,75.596,81.685,68.429,88.484,74.681
casb_edited,79.371,65.119,50.932,90.772,69.636,80.204,63.848,86.788,72.245
casb_ros,81.003,67.282,54.560,87.770,74.801,81.285,68.936,88.438,74.143


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,62.817,49.177,52.915,46.121,85.851,65.986,55.449,80.382,55.997
smote,71.377,56.250,46.423,72.130,71.225,71.678,51.210,80.098,63.814
bsmote,72.671,57.553,45.460,79.027,67.248,73.138,48.237,79.237,65.112
ros,72.220,57.149,46.613,74.429,70.429,72.429,49.585,79.216,64.684
adasyn,71.073,56.036,46.180,72.600,70.343,71.472,50.135,79.513,63.554
casb,72.844,57.876,47.511,74.434,71.620,73.027,52.688,80.258,65.360
casb_relaxed,74.055,59.230,48.029,77.669,70.823,74.246,50.497,80.127,66.642
casb_edited,74.700,59.832,46.255,85.005,65.819,75.412,50.018,80.469,67.266
casb_ros,72.316,57.265,46.952,73.953,71.141,72.547,51.036,79.605,64.791


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,63.057,50.835,58.655,45.116,89.267,67.191,63.557,84.906,56.946
smote,71.048,57.814,54.784,61.475,82.752,72.113,62.742,85.130,64.431
bsmote,70.945,57.607,54.483,61.480,82.272,71.876,59.799,84.075,64.276
ros,71.085,57.990,55.452,61.020,83.147,72.083,61.943,84.927,64.538
adasyn,71.377,58.037,54.412,62.400,82.114,72.257,60.946,85.037,64.707
casb,70.462,57.118,54.309,60.338,82.593,71.466,62.473,84.917,63.790
casb_relaxed,70.716,57.476,54.812,60.766,82.909,71.838,62.590,85.265,64.096
casb_edited,78.278,64.266,52.758,82.431,74.485,78.458,58.785,84.707,71.272
casb_ros,70.690,57.503,55.001,60.338,83.069,71.704,62.515,85.058,64.097


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,65.468,54.893,66.497,46.945,91.891,69.418,68.468,87.221,60.181
smote,79.354,66.223,56.382,80.645,78.218,79.431,67.559,87.369,72.789
bsmote,80.146,66.959,56.255,83.155,77.419,80.287,66.004,86.995,73.553
ros,79.511,66.392,56.509,81.110,78.136,79.623,67.293,87.228,72.951
adasyn,79.256,66.212,56.659,80.592,78.371,79.481,67.336,87.365,72.734
casb,78.441,65.170,55.844,78.531,78.616,78.573,68.175,87.412,71.806
casb_relaxed,78.657,65.500,56.282,78.763,78.934,78.849,68.332,87.465,72.079
casb_edited,79.556,65.797,53.920,84.799,74.807,79.803,65.527,86.597,72.676
casb_ros,79.238,65.843,55.480,81.078,77.503,79.291,67.984,87.342,72.541


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,61.696,49.699,60.898,42.828,90.282,66.555,63.125,84.455,55.697
smote,75.816,62.057,53.230,75.624,76.889,76.256,62.412,85.298,68.936
bsmote,76.330,62.521,52.820,78.154,75.455,76.805,60.230,84.654,69.425
ros,75.538,61.838,53.434,74.803,77.165,75.984,61.550,84.911,68.688
adasyn,75.921,62.143,53.015,76.590,76.248,76.419,61.634,85.236,69.032
casb,75.670,61.844,53.058,75.211,76.927,76.069,63.009,85.266,68.757
casb_relaxed,76.219,62.530,53.631,76.243,77.065,76.654,62.462,85.335,69.374
casb_edited,77.976,63.753,50.966,85.752,71.187,78.469,59.544,84.640,70.865
casb_ros,75.812,61.973,52.998,75.785,76.629,76.207,62.618,85.111,68.893


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_relaxed,81.448,adasyn,81.979,-0.531,LOSS
1,f1,casb_relaxed,67.913,adasyn,68.287,-0.374,LOSS
2,precision,casb_relaxed,55.400,smote,55.332,0.068,WIN
3,recall,casb_edited,90.772,adasyn,90.766,0.005,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,74.700,bsmote,72.671,2.029,WIN
1,f1,casb_edited,59.832,bsmote,57.553,2.279,WIN
2,precision,casb_relaxed,48.029,ros,46.613,1.415,WIN
3,recall,casb_edited,85.005,bsmote,79.027,5.978,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,78.278,adasyn,71.377,6.901,WIN
1,f1,casb_edited,64.266,adasyn,58.037,6.229,WIN
2,precision,casb_ros,55.001,ros,55.452,-0.451,LOSS
3,recall,casb_edited,82.431,adasyn,62.400,20.032,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,77.976,bsmote,76.330,1.646,WIN
1,f1,casb_edited,63.753,bsmote,62.521,1.233,WIN
2,precision,casb_relaxed,53.631,ros,53.434,0.196,WIN
3,recall,casb_edited,85.752,bsmote,78.154,7.598,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.593374,0.500000,0.800000,0.363636,0.968254,0.665945,0.720164,0.870491,0.546687,None
1,42,1,svm,smote,1.00,0.861640,0.736842,0.600000,0.954545,0.777778,0.866162,0.777200,0.921176,0.799241,None
2,42,1,svm,bsmote,1.00,0.857233,0.730435,0.591549,0.954545,0.769841,0.862193,0.757037,0.919012,0.793834,None
3,42,1,svm,ros,1.00,0.852803,0.724138,0.583333,0.954545,0.761905,0.858225,0.752645,0.908369,0.788470,None
4,42,1,svm,adasyn,1.00,0.848350,0.717949,0.575342,0.954545,0.753968,0.854257,0.799089,0.925505,0.783149,None
5,42,1,svm,casb,1.00,0.861640,0.736842,0.600000,0.954545,0.777778,0.866162,0.772490,0.917208,0.799241,None
6,42,1,svm,casb_relaxed,1.00,0.861640,0.736842,0.600000,0.954545,0.777778,0.866162,0.789094,0.922980,0.799241,None
7,42,1,svm,casb_edited,1.00,0.816497,0.677419,0.525000,0.954545,0.698413,0.826479,0.761885,0.912518,0.746958,None
8,42,1,svm,casb_ros,1.00,0.861640,0.736842,0.600000,0.954545,0.777778,0.866162,0.786387,0.918831,0.799241,None
9,42,1,knn,original,0.50,0.600325,0.467532,0.545455,0.409091,0.880952,0.645022,0.579893,0.806999,0.533929,None
